# Simulating an Online BCI — From Offline Pipeline to Real-Time Loop

*Assumes the material of all preceding notebooks, especially #2 (MI-BCI with CSP+LDA), #2.5 (evaluation methodology), and #3 (ERP classification).*

Every pipeline built so far has operated **offline**: the entire recording is available before analysis begins. Filtering uses future samples, cross-validation sees the full dataset, and classification happens after the experiment is over.

A real BCI operates **online**: data arrives sample-by-sample, decisions must be made with only past and present information, and the user is waiting — every millisecond of latency costs them. This notebook makes the transition explicit by simulating a real-time BCI loop using datasets you already know.

The simulation is not a toy. The exact same code structure — a loop that reads a buffer, filters causally, extracts features, and emits a prediction — is what runs inside production BCI systems like BCI2000, OpenViBE, and LSL-based pipelines. The only difference is that our data comes from a file instead of an amplifier.

> **No new datasets are required.** This notebook uses the PhysioNet EEGBCI dataset (notebook #2) and the MNE sample dataset (notebook #1).

## Table of contents

1. **What changes between offline and online** — The three rules of real-time processing.
2. **Causal vs acausal filtering** — Why online filters produce different results, and how to handle this.
3. **The online loop** — Architecture of a sample-by-sample BCI.
4. **Simulation 1: Motor imagery** — Streaming EEGBCI data through a trained CSP+LDA pipeline.
5. **Simulation 2: ERP detection** — Streaming the MNE sample data for auditory/visual classification.
6. **Information Transfer Rate** — The metric that matters for real users.
7. **The latency-accuracy tradeoff** — Faster decisions are less accurate; how fast is fast enough?
8. **Summary**

## 1. What changes between offline and online

Three constraints define online processing. Every decision in this notebook follows from one of these.

### Rule 1 — No future data

Offline filters (like MNE's default `raw.filter()`) are **zero-phase**: they process the signal forwards and backwards, using future samples to eliminate phase distortion. This is impossible online — at time *t*, you have only samples up to *t*. Online filters must be **causal**: the output at time *t* depends only on the present and past inputs.

Consequence: causal filters introduce **phase delay** — the filtered signal is shifted in time relative to the true neural event. This affects ERP latency measurements and the timing of BCI decisions.

### Rule 2 — Decisions must be incremental

Offline, you can epoch the entire recording, compute features on all epochs, and classify in one batch. Online, the system must emit a prediction from whatever data is currently in the buffer. This means:
- Features are computed on a **sliding window** that moves forward in time.
- The classifier is applied at each window position, producing a stream of predictions.
- The system must decide *when* it has accumulated enough evidence to commit to a decision.

### Rule 3 — Latency matters

A system that takes 10 seconds to detect a motor imagery command is useless for real-time control, even if it is 99% accurate. The relevant metric is not accuracy alone but the **Information Transfer Rate (ITR)**: how many bits of information per minute does the system deliver to the user?

| Offline metric | Online equivalent |
|---|---|
| Cross-validated accuracy | Accuracy on the current buffer |
| N/A | Decision latency (ms from cue to output) |
| N/A | Information Transfer Rate (bits/min) |

## 2. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, lfilter, lfilter_zi, sosfilt, sosfilt_zi, sosfiltfilt

import mne
from mne.datasets import eegbci, sample
from mne.channels import make_standard_montage
from mne.io import concatenate_raws, read_raw_edf
from mne.decoding import CSP

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

print("MNE:", mne.__version__)

## 3. Causal vs acausal filtering

Before building the online loop, we must understand the practical difference between causal and acausal filters. This is not a minor technical detail — it changes the shape of the signal your classifier sees.

In [ ]:
# Load a short segment of data for demonstration.
raw_fnames = eegbci.load_data(1, [4], update_path=True)
raw_demo = read_raw_edf(raw_fnames[0], preload=True)
eegbci.standardize(raw_demo)
raw_demo.set_montage(make_standard_montage("standard_1005"))

# Extract a single channel for demonstration.
sfreq = raw_demo.info["sfreq"]
data_c3 = raw_demo.get_data(picks=["C3"])[0]
t = np.arange(len(data_c3)) / sfreq

# Design a band-pass filter (7–30 Hz, Butterworth order 5).
sos = butter(5, [7, 30], btype="band", fs=sfreq, output="sos")

# Acausal (zero-phase) — what MNE does offline.
filtered_acausal = sosfiltfilt(sos, data_c3)

# Causal (forward-only) — what must be used online.
zi = sosfilt_zi(sos) * data_c3[0]
filtered_causal, _ = sosfilt(sos, data_c3, zi=zi)

print(f"Signal length: {len(data_c3)} samples ({len(data_c3)/sfreq:.1f} s)")

In [ ]:
# Compare the two filters on a 2-second window.
start, stop = int(10 * sfreq), int(12 * sfreq)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t[start:stop], data_c3[start:stop] * 1e6,
        color="lightgrey", linewidth=1, label="Unfiltered")
ax.plot(t[start:stop], filtered_acausal[start:stop] * 1e6,
        color="#2980b9", linewidth=1.5, label="Acausal (offline)")
ax.plot(t[start:stop], filtered_causal[start:stop] * 1e6,
        color="#e74c3c", linewidth=1.5, linestyle="--", label="Causal (online)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (µV)")
ax.set_title("Causal vs acausal band-pass filter (7–30 Hz)")
ax.legend()
plt.tight_layout()
plt.show()

### Interpretation

The causal filter (red dashed) is **phase-shifted** relative to the acausal filter (blue). Peaks arrive later; the waveform shape is slightly distorted. This means:

- **For MI-BCI:** The phase shift does not matter much, because CSP operates on variance (which is phase-insensitive). The main consequence is a slight delay in detecting ERD onset.
- **For ERP-BCI:** The phase shift directly affects the apparent latency of the P300 or N100. A classifier trained on acausal-filtered offline data will underperform if applied to causal-filtered online data, because the waveform shape has changed.

📖 **Practical rule:** Train your classifier on **causal-filtered** data if it will be deployed online. Never train offline (acausal) and deploy online (causal) — the mismatch degrades performance.

## 4. The online loop — architecture

The core of any online BCI is a loop that repeats at a fixed update rate:

```
while running:
    1. Read new samples from the amplifier (or simulated stream)
    2. Append them to a circular buffer
    3. Apply causal filter to the buffer
    4. If enough data has accumulated:
       a. Extract features from the current window
       b. Classify
       c. Output the prediction (or accumulate evidence)
    5. Advance the window
```

The key parameters are:
- **Buffer length:** How many seconds of past data to keep. Longer buffers give more context but increase latency.
- **Update interval:** How often to emit a new prediction (e.g., every 100 ms).
- **Window length:** How many seconds of data to use for each feature extraction (e.g., 1 s for MI, 0.6 s for ERP).

## 5. Simulation 1 — Motor imagery online loop

We train a CSP+LDA classifier offline (as in notebook #2), then simulate feeding data sample-by-sample and watching the classifier's predictions evolve over time within each trial.

### 5.1 Offline training phase

In [ ]:
# Load and preprocess — identical to notebook #2.
subject = 1
runs_train = [4, 8]     # Use runs 4 and 8 for training
runs_test  = [12]       # Hold out run 12 for the simulated session

def load_mi_raw(subject, runs):
    raw_fnames = eegbci.load_data(subject, runs, update_path=True)
    raw = concatenate_raws([read_raw_edf(f, preload=True) for f in raw_fnames])
    eegbci.standardize(raw)
    raw.set_montage(make_standard_montage("standard_1005"))
    raw.filter(l_freq=7.0, h_freq=30.0)
    return raw

# Training data.
raw_train = load_mi_raw(subject, runs_train)
events_train, eid_train = mne.events_from_annotations(raw_train)
eid_lr = {k: v for k, v in eid_train.items() if v in (2, 3)}

epochs_train = mne.Epochs(raw_train, events_train, eid_lr,
                          tmin=0.5, tmax=3.5, baseline=None,
                          picks="eeg", preload=True)

X_train = epochs_train.get_data(copy=False) * 1e6
y_train = epochs_train.events[:, -1]

# Train the classifier.
csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)
lda = LinearDiscriminantAnalysis()
clf = Pipeline([("CSP", csp), ("LDA", lda)])
clf.fit(X_train, y_train)

train_acc = clf.score(X_train, y_train)
print(f"Training accuracy: {train_acc:.1%}")
print(f"Training trials:   {len(y_train)}")

### 5.2 Simulated online session

Now we take the held-out run (run 12), and instead of epoching it all at once, we **stream** it: we advance through the continuous recording in small steps, maintaining a sliding window, and apply the trained classifier at each step.

In [ ]:
# Load the test run — but do NOT epoch it. We process it as continuous data.
raw_test = load_mi_raw(subject, runs_test)
events_test, eid_test = mne.events_from_annotations(raw_test)
sfreq = raw_test.info["sfreq"]
data_test = raw_test.get_data(picks="eeg") * 1e6   # (channels, samples)
n_channels, n_samples = data_test.shape

# Extract true event times and labels.
true_events = [(ev[0], ev[2]) for ev in events_test if ev[2] in (2, 3)]

print(f"Continuous test data: {n_channels} channels × {n_samples} samples ({n_samples/sfreq:.1f} s)")
print(f"Events in test run: {len(true_events)} imagery trials")

In [ ]:
# ── The online simulation loop. ──

window_sec = 2.0        # Feature extraction window (seconds)
step_sec   = 0.1        # Update interval (seconds) — emit a prediction every 100 ms
window_samp = int(window_sec * sfreq)
step_samp   = int(step_sec * sfreq)

# Storage for the prediction stream.
pred_times = []         # Time of each prediction (in samples)
pred_labels = []        # Predicted class at each time point
pred_probs = []         # Classifier confidence (LDA decision function)

# Slide through the recording.
for start in range(0, n_samples - window_samp, step_samp):
    end = start + window_samp
    window = data_test[:, start:end]  # (channels, window_samples)

    # CSP+LDA expects (n_trials, n_channels, n_times) → add a trial dimension.
    X_window = window[np.newaxis, :, :]

    # Classify.
    label = clf.predict(X_window)[0]
    prob = clf.decision_function(X_window)[0]

    pred_times.append(end)       # prediction refers to the END of the window
    pred_labels.append(label)
    pred_probs.append(prob)

pred_times = np.array(pred_times)
pred_labels = np.array(pred_labels)
pred_probs = np.array(pred_probs)

print(f"Predictions emitted: {len(pred_times)}")
print(f"Update rate: every {step_sec*1000:.0f} ms")
print(f"Window length: {window_sec} s")

In [ ]:
# Visualise the prediction stream for the first few trials.
fig, ax = plt.subplots(figsize=(14, 5))

# Plot the classifier's confidence over time.
t_pred = pred_times / sfreq
ax.plot(t_pred, pred_probs, color="steelblue", linewidth=0.8, alpha=0.8)
ax.axhline(0, color="grey", linewidth=1, linestyle="-")
ax.fill_between(t_pred, pred_probs, 0,
                where=pred_probs > 0, alpha=0.15, color="#e74c3c", label="Predicts RIGHT")
ax.fill_between(t_pred, pred_probs, 0,
                where=pred_probs <= 0, alpha=0.15, color="#2980b9", label="Predicts LEFT")

# Mark true events.
for ev_sample, ev_label in true_events:
    ev_time = ev_sample / sfreq
    color = "#2980b9" if ev_label == 2 else "#e74c3c"
    name = "LEFT cue" if ev_label == 2 else "RIGHT cue"
    ax.axvline(ev_time, color=color, linewidth=1.5, linestyle="--", alpha=0.7)

ax.set_xlabel("Time (s)")
ax.set_ylabel("Classifier confidence\n← LEFT    RIGHT →")
ax.set_title("Simulated online MI-BCI — classifier output over time")
ax.legend(loc="upper right")

# Zoom into the first 60 seconds.
ax.set_xlim(0, 60)
plt.tight_layout()
plt.show()

### Interpretation

The plot shows the classifier's **continuous output** — a signed confidence score that swings positive (right imagery) or negative (left imagery) as data streams through. The vertical dashed lines mark the true imagery cues.

Several phenomena are visible that offline analysis hides:

1. **Transient errors.** The classifier fluctuates between classes during the transition between rest and imagery. This is not a bug — the sliding window temporarily contains a mix of rest and imagery data.

2. **Decision lag.** After the cue (dashed line), it takes 0.5–1.5 seconds before the classifier stabilises on the correct prediction. This is the **decision latency** — the time the user must wait before the system responds.

3. **Between-trial noise.** During rest periods, the classifier output is essentially random — it has no meaningful signal to classify. A real system must know when the user is *not* performing imagery and suppress output during these intervals.

These phenomena are invisible in offline cross-validated accuracy. A system with 75% offline accuracy may have 90% accuracy *within imagery periods* but 50% (chance) during rest — and the user experience depends heavily on how the system handles the transitions.

### 5.3 Evaluating the online session

We evaluate the simulated session by checking the classifier's prediction during the known imagery intervals (0.5–3.5 s after each cue).

In [ ]:
# For each true event, check what the classifier predicted during the imagery interval.
correct = 0
total = 0
latencies = []  # time from cue to first correct prediction (seconds)

for ev_sample, ev_label in true_events:
    # Imagery interval: 0.5 to 3.5 s after cue.
    t_start = ev_sample + int(0.5 * sfreq)
    t_end   = ev_sample + int(3.5 * sfreq)

    # Find all predictions within this interval.
    mask = (pred_times >= t_start) & (pred_times <= t_end)
    if not np.any(mask):
        continue

    # Majority vote: the most frequent prediction wins.
    preds_in_window = pred_labels[mask]
    majority = np.bincount(preds_in_window.astype(int)).argmax()
    # Map back: LDA predicts the original labels (2 or 3)
    is_correct = (majority == ev_label)
    correct += int(is_correct)
    total += 1

    # Latency: time from cue to first correct prediction.
    times_in_window = pred_times[mask]
    labels_in_window = pred_labels[mask]
    correct_mask = labels_in_window == ev_label
    if np.any(correct_mask):
        first_correct = times_in_window[correct_mask][0]
        latency_s = (first_correct - ev_sample) / sfreq
        latencies.append(latency_s)

online_acc = correct / total if total > 0 else 0
mean_latency = np.mean(latencies) if latencies else float('nan')

print(f"Online session results:")
print(f"  Trials evaluated:     {total}")
print(f"  Majority-vote accuracy: {online_acc:.1%}")
print(f"  Mean latency to first correct: {mean_latency:.2f} s")

## 6. Simulation 2 — ERP detection loop

The MI loop classifies **ongoing** activity within a sustained window. ERP detection is different: the system must detect a **transient** response to a discrete stimulus. The online logic changes accordingly — instead of sliding a window continuously, the system epochs around stimulus markers and classifies each epoch.

In [ ]:
# Load the MNE sample dataset.
from mne.datasets import sample as mne_sample
from mne.decoding import Vectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

data_path = mne_sample.data_path()
raw_erp = mne.io.read_raw_fif(data_path / "MEG" / "sample" / "sample_audvis_raw.fif",
                                preload=True)
events_erp = mne.find_events(raw_erp, stim_channel="STI 014")
raw_erp.pick("eeg")

# Causal filter — critical for online simulation.
# MNE's filter with method='iir' and phase='minimum' gives a causal filter.
raw_erp.filter(l_freq=0.1, h_freq=40.0, method="iir",
               iir_params=dict(order=5, ftype="butter"), phase="minimum")
raw_erp.set_eeg_reference("average", projection=True)
raw_erp.apply_proj()

print(f"ERP data loaded: {raw_erp.info['nchan']} channels, {raw_erp.times[-1]:.0f} s")

In [ ]:
# Split events: first half for training, second half for simulation.
eid_erp = {"auditory": 1, "visual": 3}  # collapse left/right
mask = np.isin(events_erp[:, -1], [1, 2, 3, 4])
events_av = events_erp[mask]
# Relabel: 1,2 → auditory; 3,4 → visual
events_av[np.isin(events_av[:, -1], [1, 2]), -1] = 1
events_av[np.isin(events_av[:, -1], [3, 4]), -1] = 3

n_ev = len(events_av)
split = n_ev // 2
events_train_erp = events_av[:split]
events_test_erp = events_av[split:]

# Train on first half.
epochs_train_erp = mne.Epochs(raw_erp, events_train_erp, {"aud": 1, "vis": 3},
                               tmin=0.0, tmax=0.5, baseline=None,
                               picks="eeg", preload=True, reject=dict(eeg=100e-6))
X_tr_erp = epochs_train_erp.get_data(copy=False)
y_tr_erp = (epochs_train_erp.events[:, -1] == 3).astype(int)  # 0=aud, 1=vis

clf_erp = make_pipeline(Vectorizer(), StandardScaler(),
                         LogisticRegression(solver="liblinear"))
clf_erp.fit(X_tr_erp, y_tr_erp)
print(f"ERP classifier trained on {len(y_tr_erp)} trials")
print(f"Training accuracy: {clf_erp.score(X_tr_erp, y_tr_erp):.1%}")

In [ ]:
# Simulate online ERP detection: for each test event, epoch on the fly and classify.
data_erp = raw_erp.get_data()  # (channels, samples)
sfreq_erp = raw_erp.info["sfreq"]
epoch_samples = int(0.5 * sfreq_erp)

online_preds = []
online_true = []
online_latencies_erp = []

for ev in events_test_erp:
    ev_sample, _, ev_label = ev
    if ev_label not in (1, 3):
        continue

    # In a real system, this epoch would be extracted as data arrives.
    start = ev_sample
    end = start + epoch_samples
    if end > data_erp.shape[1]:
        continue

    window = data_erp[:, start:end][np.newaxis, :, :]

    # Reject if amplitude too high (online artifact rejection).
    if np.any(np.abs(window) > 100e-6):
        continue

    pred = clf_erp.predict(window)[0]
    true = 1 if ev_label == 3 else 0
    online_preds.append(pred)
    online_true.append(true)

online_preds = np.array(online_preds)
online_true = np.array(online_true)
erp_online_acc = np.mean(online_preds == online_true)

print(f"\nOnline ERP detection:")
print(f"  Trials evaluated: {len(online_true)}")
print(f"  Online accuracy:  {erp_online_acc:.1%}")
print(f"  Decision latency: {0.5:.1f} s (fixed — one epoch window)")

## 7. Information Transfer Rate

Accuracy alone does not capture online BCI performance. A system that is 90% accurate but takes 30 seconds per decision is worse than one that is 70% accurate with 2-second decisions. The standard metric that combines accuracy and speed is the **Information Transfer Rate (ITR)**.

For a system with *N* classes, accuracy *P*, and *T* seconds per decision:

$$\text{ITR} = \frac{60}{T} \left[ \log_2 N + P \log_2 P + (1-P) \log_2 \frac{1-P}{N-1} \right] \quad \text{bits/min}$$

In [ ]:
def itr_bits_per_min(n_classes, accuracy, trial_duration_s):
    """Compute Information Transfer Rate in bits per minute."""
    P = np.clip(accuracy, 1e-10, 1 - 1e-10)  # avoid log(0)
    N = n_classes
    bits = np.log2(N) + P * np.log2(P) + (1 - P) * np.log2((1 - P) / (N - 1))
    return (60.0 / trial_duration_s) * bits


# Compare different operating points.
print("Information Transfer Rate comparison:")
print(f"{'System':<35s} {'Acc':>6s} {'Trial':>6s} {'ITR':>10s}")
print("-" * 60)

scenarios = [
    ("MI-BCI (offline, full trial)",   2, 0.75,  4.0),
    ("MI-BCI (online, 2s window)",     2, 0.70,  2.0),
    ("MI-BCI (online, 1s window)",     2, 0.62,  1.0),
    ("P300 speller (1 sequence)",      36, 0.40,  2.5),
    ("P300 speller (5 sequences)",     36, 0.85, 12.5),
    ("SSVEP (typical)",               4, 0.90,  3.0),
]

for name, n, acc, t in scenarios:
    rate = itr_bits_per_min(n, acc, t)
    print(f"{name:<35s} {acc:>5.0%} {t:>5.1f}s {rate:>8.1f} b/m")

### Interpretation

ITR reveals tradeoffs that accuracy alone hides:

- Shortening the MI window from 4 s to 1 s reduces accuracy (75% → 62%) but may *increase* ITR because decisions are faster.
- The P300 speller with 1 sequence is fast but inaccurate; with 5 sequences it is accurate but slow. The optimal operating point depends on the user.
- SSVEP systems typically achieve the highest ITR because they combine high accuracy with moderate speed.

📖 **ITR is the metric that matters for the user.** A BCI with high accuracy but low ITR is frustrating. A BCI with moderate accuracy but high ITR may feel responsive and usable.

## 8. The latency-accuracy tradeoff

Using the MI simulation from Section 5, we can sweep the window length and measure how accuracy and latency trade off.

In [ ]:
# Sweep window lengths from 0.5 s to 3.0 s.
window_lengths = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
sweep_results = []

for w_sec in window_lengths:
    w_samp = int(w_sec * sfreq)
    correct = 0
    total = 0
    for ev_sample, ev_label in true_events:
        # Take a single window starting 0.5 s after cue.
        start = ev_sample + int(0.5 * sfreq)
        end = start + w_samp
        if end > n_samples:
            continue
        window = data_test[:, start:end][np.newaxis, :, :]
        pred = clf.predict(window)[0]
        correct += int(pred == ev_label)
        total += 1

    acc = correct / total if total > 0 else 0
    trial_time = 0.5 + w_sec  # cue-to-decision time
    rate = itr_bits_per_min(2, acc, trial_time)
    sweep_results.append((w_sec, acc, trial_time, rate))
    print(f"Window {w_sec:.1f}s: acc={acc:.1%}, latency={trial_time:.1f}s, ITR={rate:.1f} b/m")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

wins = [r[0] for r in sweep_results]
accs = [r[1] for r in sweep_results]
itrs = [r[3] for r in sweep_results]

ax1.plot(wins, accs, "o-", color="#2980b9", linewidth=2)
ax1.set_xlabel("Window length (s)")
ax1.set_ylabel("Accuracy")
ax1.set_title("Longer windows → higher accuracy")
ax1.axhline(0.5, color="grey", linestyle="--")

ax2.plot(wins, itrs, "o-", color="#e74c3c", linewidth=2)
ax2.set_xlabel("Window length (s)")
ax2.set_ylabel("ITR (bits/min)")
ax2.set_title("The optimal window maximises ITR")

plt.suptitle("Latency-accuracy tradeoff — MI-BCI, subject 1", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

best_idx = np.argmax(itrs)
print(f"\nOptimal window: {wins[best_idx]:.1f} s "
      f"(accuracy={accs[best_idx]:.1%}, ITR={itrs[best_idx]:.1f} bits/min)")

### Interpretation

The left panel shows the expected pattern: longer windows capture more ERD and yield higher accuracy. But the right panel reveals that ITR peaks at an intermediate window length. Beyond that point, the accuracy gain is too small to compensate for the additional latency.

This is the central engineering tradeoff of online BCIs. The "best" window length depends on the application: a communication aid for a locked-in patient may tolerate longer latencies to maximise accuracy; a gaming interface may prioritise speed even at the cost of more errors.

---

❓ **Exercise.** Modify the MI simulation to implement a **confidence threshold**: the system only commits to a prediction when the LDA decision function exceeds a threshold (e.g., |d| > 0.5). Below the threshold, it outputs "uncertain" and waits for more data. How does this affect the accuracy-latency tradeoff?

## 9. Summary

### Key concepts

1. **Causal filters** are mandatory online. They introduce phase delay and slightly change the signal shape. Train on causally-filtered data if deploying online.

2. **The online loop** reads data incrementally, maintains a sliding buffer, and emits predictions at a fixed update rate. The architecture is the same for MI and ERP systems; the windowing strategy differs.

3. **Offline accuracy overestimates online performance.** Transition periods, between-trial noise, and causal-filter effects all degrade real-time performance relative to epoch-and-classify evaluation.

4. **ITR is the user-facing metric.** It combines accuracy and speed into a single number: bits of information transferred per minute. Optimising accuracy alone may produce a system that is correct but unusably slow.

5. **The latency-accuracy tradeoff has an optimum.** Longer windows increase accuracy but also increase decision time. ITR reveals the sweet spot.